# Kiểm chứng full-model Task 1 → Task 2

Ngày tạo: 10/10/2026, UTC+7. Real run dùng checkpoint Task 1 từ notebook 01 trên cùng GPU 4090. Mục tiêu: dựng teacher frozen, khởi tạo prompts task mới, chạy distillation/PPG với prototype memory và loader chỉ chứa current-task GT. Đây là smoke chuyển task, không phải baseline Task 2 hội tụ.

[Runbook](../../docs/project/TRAINING_NOTEBOOK_RUNBOOK.md) mô tả tmux, log và resume. Mọi artifacts dưới data. Mặc định dry run CPU/tiny ResNet18/6 labels; giữ cùng settings/campaign của notebook 01.


## 1. Parent checkpoint và subset current-task
Notebook yêu cầu pilot đã hoàn tất. Copy riêng task_final.pth của Task 1 vào thư mục transition; không copy predictions Task 1 để tránh trộn annotation checksums của hai phạm vi. Derived Task 2 train subset mặc định tối đa 16 ảnh, 2 ảnh val/test; train JSON chỉ có nhãn task mới. Không sửa release/holdout. Parent hash và subset hashes được khóa trong plan.json.


In [ ]:
import json, os, sys
from pathlib import Path
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file())
sys.path.insert(0, str(REPO))
from autocheckout.io import load_json
from autocheckout.training_workflow import prepare, run_phase, resolve
SETTINGS_PATH = resolve(os.environ.get('AUTOCHECKOUT_TRAINING_SETTINGS', 'configs/training/notebooks.json'))
SETTINGS = load_json(SETTINGS_PATH)
PHASE = 'transition'
PLAN = prepare(SETTINGS, PHASE)
print(json.dumps({k: PLAN[k] for k in ['phase_dir', 'dry_run', 'epochs', 'task_dir', 'parent_checkpoint_hashes', 'batch_size', 'effective_batch']}, indent=2))


## 2. Task 2 qua CLI thật
Runner bắt đầu ở task 2. Teacher dựng từ parent checkpoint trước khi khởi tạo prompts mới; telemetry phải xác nhận teacher_present, teacher_frozen và teacher_forward_calls > 0. Ghi số pseudo labels được thêm và lớp thiếu prototype; số pseudo labels có thể bằng 0 ở smoke và phải được báo đúng. Resume dùng last.ckpt của task 2, giữ parent checkpoint không đổi.


In [ ]:
SUMMARY = run_phase(PLAN)
for case in SUMMARY['cases']:
    for task in case['tasks']:
        assert task['teacher']['teacher_present'] and task['teacher']['teacher_frozen']
        print(case['experiment'], json.dumps(task, indent=2))
print('Status:', SUMMARY['status'], 'dry_run:', SUMMARY['dry_run'])
print('Source data hashes unchanged:', SUMMARY['input_hashes_unchanged'])


## 3. Kết luận và chọn baseline budget
COMPLETED xác nhận đường chuyển task đã chạy và đủ artifacts trong phạm vi subset. Metrics transition là diagnostics; không dùng thay kết quả Task 2 đầy đủ. Dry run không chứng nhận full ConvNeXt/GPU 4090.

Sau real pilot và transition, đặt baseline_epochs trong settings dựa trên val curves. Notebook 03 tạo baseline mới từ pretrained ban đầu, train lại Task 1 đủ budget rồi lần lượt Task 2–5; không tái sử dụng Task 1 pilot ngắn như một Task 1 đã hội tụ.
